<a href="https://colab.research.google.com/github/majdsabha/Prediction-of-Product-Sales/blob/main/Project_1_Part_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Project 1 - Part 6
-Stucent Name : Majd Sabha

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
fpath = '/content/drive/MyDrive/AXSOSACADEMY/AXSOSACADEMY/01-Fundamentals/Week02/Data/sales_predictions_2023.csv'
df = pd.read_csv(fpath)

df.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [9]:
# Import libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline, make_pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
# Check dataset shape
df.shape

(8523, 12)

In [4]:
# Check data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   object 
 1   Item_Weight                7060 non-null   float64
 2   Item_Fat_Content           8523 non-null   object 
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   object 
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   object 
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                6113 non-null   object 
 9   Outlet_Location_Type       8523 non-null   object 
 10  Outlet_Type                8523 non-null   object 
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(4), int64(1), object(7)
memory usage: 799.2+ KB


In [5]:
# Check missing values
df.isna().sum()

,0
Item_Identifier,0
Item_Weight,1463
Item_Fat_Content,0
Item_Visibility,0
Item_Type,0
Item_MRP,0
Outlet_Identifier,0
Outlet_Establishment_Year,0
Outlet_Size,2410
Outlet_Location_Type,0


In [6]:
# Summary statistics
df.describe()

,Item_Weight,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Item_Outlet_Sales
count,7060.000000,8523.000000,8523.000000,8523.000000,8523.000000
mean,12.857645,0.066132,140.992782,1997.831867,2181.288914
std,4.643456,0.051598,62.275067,8.371760,1706.499616
min,4.555000,0.000000,31.290000,1985.000000,33.290000
25%,8.773750,0.026989,93.826500,1987.000000,834.247400
50%,12.600000,0.053931,143.012800,1999.000000,1794.331000
75%,16.850000,0.094585,185.643700,2004.000000,3101.296400
max,21.350000,0.328391,266.888400,2009.000000,13086.964800


In [7]:
# Define features and target

X = df.drop(columns='Item_Outlet_Sales')
y = df['Item_Outlet_Sales']

In [10]:
# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(X, y,random_state=42)

In [11]:
# Create column selectors

num_selector = make_column_selector(dtype_include='number')
cat_selector = make_column_selector(dtype_include='object')

In [12]:
# Create numeric preprocessing pipeline

num_pipe = make_pipeline( SimpleImputer(strategy='mean'),StandardScaler())

# Create categorical preprocessing pipeline

cat_pipe = make_pipeline(SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore',sparse_output=False))

In [13]:
# Create preprocessing ColumnTransformer

preprocessor = ColumnTransformer([  ('num', num_pipe, num_selector), ('cat', cat_pipe, cat_selector)])

In [14]:
# Fit the preprocessor on training data only

preprocessor.fit(X_train)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer()),
                                                 ('standardscaler',
                                                  StandardScaler())]),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x7918929a4f50>),
                                ('cat',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x791892a54b30>)])

In [16]:
# Transform the training and testing data

X_train_tf = preprocessor.transform(X_train)
X_test_tf = preprocessor.transform(X_test)


#1.Bulid linear Regressuion model



In [17]:
# Build Linear Regression model

lin_reg = LinearRegression()
lin_reg

LinearRegression()

In [18]:
# Fit model on transformed training data

lin_reg.fit(X_train_tf, y_train)

LinearRegression()

In [20]:
# Create helper function for regression metrics

def regression_metrics(y_true, y_pred, label='', verbose=True,
                       output_dict=False):

    # Get metrics
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r_squared = r2_score(y_true, y_pred)

    # Print results
    if verbose == True:
        header = "-" * 60
        print(header, f"Regression Metrics: {label}", header, sep="\n")
        print(f"- MAE = {mae:,.3f}")
        print(f"- MSE = {mse:,.3f}")
        print(f"- RMSE = {rmse:,.3f}")
        print(f"- R^2 = {r_squared:.3f}")

    # Return dictionary if needed
    if output_dict == True:
        metrics = {
            'Label': label,
            'MAE': mae,
            'MSE': mse,
            'RMSE': rmse,
            'R^2': r_squared
        }
        return metrics

In [19]:
# Create helper function to evaluate regression models

def evaluate_regression(reg, X_train, y_train, X_test, y_test,
                        verbose=True, output_frame=False):

    # Get predictions for training data
    y_train_pred = reg.predict(X_train)

    # Get metrics for training data
    results_train = regression_metrics(
        y_train,
        y_train_pred,
        verbose=verbose,
        output_dict=output_frame,
        label='Training Data'
    )

    print()

    # Get predictions for test data
    y_test_pred = reg.predict(X_test)

    # Get metrics for test data
    results_test = regression_metrics(
        y_test,
        y_test_pred,
        verbose=verbose,
        output_dict=output_frame,
        label='Test Data'
    )

    # Store results in a dataframe if output_frame is True
    if output_frame:
        results_df = pd.DataFrame([results_train, results_test])
        results_df = results_df.set_index('Label')
        results_df.index.name = None
        return results_df.round(3)

In [21]:
# Evaluate Linear Regression model

evaluate_regression( lin_reg,X_train_tf, y_train, X_test_tf, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 735.569
- MSE = 971,169.101
- RMSE = 985.479
- R^2 = 0.672

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 967.198
- MSE = 1,702,124.948
- RMSE = 1,304.655
- R^2 = 0.383


##Compare the training vs. test R² values and answer the question: to what extent is this model overfit/underfit?


-The Linear Regression model shows overfitting because the training R² score (0.672) is much higher than the test R² score (0.383).

#2.Bulid Random Forest model

In [24]:
# Create a default Random Forest model

rf = RandomForestRegressor(random_state=42)
rf

RandomForestRegressor(random_state=42)

In [25]:
# Fit model on transformed training data

rf.fit(X_train_tf, y_train)

RandomForestRegressor(random_state=42)

In [27]:
# Evaluate default Random Forest



evaluate_regression( rf,X_train_tf, y_train, X_test_tf,y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 296.898
- MSE = 183,359.734
- RMSE = 428.205
- R^2 = 0.938

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 772.986
- MSE = 1,239,893.293
- RMSE = 1,113.505
- R^2 = 0.551


##Compare the training vs. test R² values and answer the question: to what extent is this model overfit/underfit?

-The Random Forest model shows overfitting because the training R² (0.938) is much higher than the test R² (0.551).

##Compare this model's performance to the linear regression model: which model has the best test scores?

-The Random Forest model has the best test score because its test R² (0.551) is higher than the Linear Regression test R² (0.383).

#3.Create GridSearchCV for Random Forest

In [28]:
# Define parameter grid for Random Forest
rf_param_grid = {'n_estimators': [50, 100, 200], 'max_depth': [None, 5, 10]}

In [30]:
# Create GridSearchCV for Random Forest
rf_gridsearch = GridSearchCV( RandomForestRegressor(random_state=42),rf_param_grid,cv=5, scoring='r2', n_jobs=-1, verbose=1)

In [31]:
# Fit the GridSearchCV on the training data
rf_gridsearch.fit(X_train_tf, y_train)

Fitting 5 folds for each of 9 candidates, totalling 45 fits


GridSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 5, 10],
                         'n_estimators': [50, 100, 200]},
             scoring='r2', verbose=1)

In [34]:
# Display best parameters

rf_gridsearch.best_params_

{'max_depth': 5, 'n_estimators': 50}

##After determining the best parameters from your GridSearch, fit and evaluate a final best model on the entire training set.

In [35]:
# Get the best Random Forest model

best_rf = rf_gridsearch.best_estimator_

In [36]:
# Evaluate the best Random Forest model

evaluate_regression( best_rf, X_train_tf, y_train, X_test_tf,  y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 755.528
- MSE = 1,147,542.837
- RMSE = 1,071.234
- R^2 = 0.612

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 729.285
- MSE = 1,097,011.663
- RMSE = 1,047.383
- R^2 = 0.602


##Compare your tuned model to your default Random Forest: did the performance improve?

-Yes, the performance improved because the tuned Random Forest achieved a higher test R² score (0.602) than the default model (0.551).

#4. You now have tried several different models on your data set. You need to determine which model to implement.

###1.Overall, which model do you recommend?
*   Tuned Random Forest Model.





###2. Justify your recommendation.


*   the tuned Random Forest model because it achieved the highest test R² score and better prediction accuracy than the other models.




###3.Interpret your model's performance based on R-squared in a way that your non-technical stakeholder can understand.


*   I recommend the tuned Random Forest model because it performed best on new data. Its training score was 0.612 and its test score was 0.602, which are very close. This means the model is not overfitting and can provide acceptable predictions for future sales.
  



###4.Select another regression metric (RMSE / MAE / MSE) to express the performance of your model to your stakeholder.

*  RMSE




###5. Include why you selected this metric.

*   I selected RMSE because it shows the average prediction error. The model’s RMSE is 1,047, meaning predictions are off by about 1,047 sales units on average.



###6. Compare the training vs. test scores and answer the question:to what extent is this model overfit/underfit?


*   The model shows very little overfitting because the training and test scores are very close.


